# 因子 × label 逐年 IC / ICIR（Pearson / Spearman）

参考 `factor_ic.py`。给一个**路径前缀**（如 `obi`），脚本自动扫描
`output/factors/<prefix>/` 下的各个变式子目录，用你写的 `select_factor(name)`
挑出本次要算的那些。

**对齐口径**：因子每个交易日 9 行（含 15:00 收盘半小时），label 每日 8 行。
按 `exchtime` 取交集 → 15:00 那行自动丢弃，只留和 label 对齐的 **前 8 个时点**
（含 09:30 开盘），与 label 一一对应。

**4 个指标**（逐年统计：先每个时点算截面相关，再按年聚合，丢弃非有限值；口径同 `factor_ic.py`）：

| key | 相关 | 统计量 |
|---|---|---|
| `pearson_ic`    | Pearson  | IC = 年内截面相关的均值 |
| `spearman_ic`   | Spearman | IC = 年内截面相关的均值 |
| `pearson_icir`  | Pearson  | ICIR = 年内均值 / 年内 std(ddof=1) |
| `spearman_icir` | Spearman | ICIR = 年内均值 / 年内 std(ddof=1) |

**市值 / 行业中性化（可选，`NEUTRALIZE`）**：置为 `"size"` / `"industry"` / `"both"`
后，每个截面先把因子对 `[1, z(log流通市值), 中证一级行业哑元]` 做 OLS、取**残差**，
再和 label 算 IC —— 即剔除市值、行业带来的截面暴露。暴露数据（见 `factor_neutralize.py`）：
行业取 JYDB `LC_CSIIndustry` 中证一级（覆盖至今）；流通市值 = 无限售流通股
`NonResiSharesJY`（快照冻结于 2025-04，按最近快照 carry-forward）× 当日 NAS 原始 `Close`，
与 JYDB `NegotiableMV` 交叉验证 `corr≈0.9997`。首次运行触库+读盘构建 `EXPO_CACHE`，之后复用。

结果存进变量 `results`（`results['ic']` 即 4 张逐年表）。作 4 张逐年柱状图；
再用 `print_ic_tables([...])` 传入因子名 list，输出 4 张 markdown 表。

In [ ]:
# ============================ 配置 ============================
import os

FACTOR_ROOT = "output/factors"   # 因子输出根目录
LABEL_DIR   = "label/1d"         # label 目录（每日 8 行宽表）
PREFIX      = os.environ.get("IC_PREFIX", "obi")   # 因子路径前缀 → output/factors/<PREFIX>

MIN_COUNT   = 30                 # 单截面最少有效样本数，不足记 NaN
START       = os.environ.get("IC_START") or None   # 起始日期 YYYYMMDD（含），None=不限
END         = os.environ.get("IC_END")   or None   # 结束日期 YYYYMMDD（含），None=不限
N_WORKERS   = os.cpu_count()   # 并行【进程】数（多进程绕开 GIL；加线程没用，进程才快）

INCLUDE_ALL = False              # True 时在逐年之外再加一组/行 "ALL"（全期汇总）

# ---- 市值 / 行业中性化（可选）----
# None=不做；"size"=只市值；"industry"=只行业；"both"=市值+行业
# 中性化 = 每个截面把因子对 [1, z(log流通市值), 行业哑元] 回归取残差，再和 label 算 IC。
# NEUTRALIZE   = os.environ.get("IC_NEUTRALIZE") or None
NEUTRALIZE   = "both"
INDUSTRY_COL = "FstIndustryCSI"                 # 中证一级(≈11类)；二级用 "SndIndustryCSI"
EXPO_CACHE   = f"results/exposures_{INDUSTRY_COL}.pkl"  # 统一暴露缓存：与因子无关，仅按行业口径隔离
# 暴露（行业/市值）只取决于交易日范围、与因子本身无关，故所有因子批次共享这一份缓存，一直复用。
# 市值口径：无限售流通股(JYDB NonResiSharesJY, 冻结于2025-04的快照 carry-forward)
#           × 当日 NAS refdata 原始 Close。与 JYDB NegotiableMV 交叉验证 corr≈0.9997。
# 市值已在建缓存时算好并入 EXPO_CACHE，运行期不再读 NAS。
# 本次日期若超出缓存，只【增量】补缺的交易日、不重算历史；换 INDUSTRY_COL 口径自动换/重建缓存。

## 1. 因子筛选函数
改这个函数决定本次算哪些变式；`name` = `output/factors/<PREFIX>/` 下的子文件夹名。

In [ ]:
def select_factor(name: str) -> bool:
    """返回 True 表示本次纳入计算。
    例：
        return True               # 全部变式
        return "bid" in name      # 只算名字里含 'bid' 的
        return name in {"mean", "twa"}
    """
    return True

## 2. 目录发现 + 读文件

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata


def is_date(name: str) -> bool:
    return len(name) == 8 and name.isdigit()


def date_dirs(path: str):
    if not os.path.isdir(path):
        return []
    return sorted(d for d in os.listdir(path)
                  if is_date(d) and os.path.isdir(os.path.join(path, d)))


def csv_in(date_dir: str):
    for f in sorted(os.listdir(date_dir)):
        if f.endswith(".csv"):
            return os.path.join(date_dir, f)
    return None


def discover_factors(prefix: str, selector) -> dict:
    """返回 {factor_name: series_dir}，只保留 selector(name)==True 的。
    两种布局：
      - <root>/<prefix> 直接含日期目录        → 单因子，名字用 prefix
      - <root>/<prefix>/<变式>/<date>/...csv  → 每个变式一个因子
    """
    root = os.path.join(FACTOR_ROOT, prefix)
    if not os.path.isdir(root):
        raise FileNotFoundError(f"因子目录不存在: {root}")
    found = {}
    if date_dirs(root):                                  # 单因子布局
        name = os.path.basename(root.rstrip("/"))
        if selector(name):
            found[name] = root
    else:                                                # 多变式布局
        for e in sorted(os.listdir(root)):
            p = os.path.join(root, e)
            if os.path.isdir(p) and date_dirs(p) and selector(e):
                found[e] = p
    return found


def _load(path: str):
    """读宽表 → (exchtime[int64], symbols[list], matrix[rows×nsym, float])。"""
    df = pd.read_csv(path)
    ex = df["exchtime"].to_numpy(dtype=np.int64)
    syms = list(df.columns[2:])                          # 跳过 exchtime, localtime
    mat = df[syms].to_numpy(dtype=float)                 # 'nan' → NaN
    return ex, syms, mat

## 3. 单截面相关 + 单日处理
`slice_idx` 由 label 当日 `exchtime` 升序排名给出（0 = 最早 = 09:30 开盘）。
因子里的 15:00 收盘行不在 label → 自动丢弃；对因子缺行也稳健。

In [ ]:
import factor_neutralize as fn   # 市值/行业中性化（可选）


def _corr_pair(f: np.ndarray, l: np.ndarray, min_count: int):
    """单截面 → (pearson, spearman)；样本不足或零方差记 NaN。"""
    mask = np.isfinite(f) & np.isfinite(l)
    n = int(mask.sum())
    if n < min_count:
        return np.nan, np.nan
    fm, lm = f[mask], l[mask]
    pear = (np.nan if fm.std() == 0 or lm.std() == 0
            else float(np.corrcoef(fm, lm)[0, 1]))
    fr, lr = rankdata(fm), rankdata(lm)                  # 平均秩，处理并列
    spear = (np.nan if fr.std() == 0 or lr.std() == 0
             else float(np.corrcoef(fr, lr)[0, 1]))
    return pear, spear


def process_date(date, label_file, factor_files, min_count,
                 neutralize=None, expo=None):
    """一个交易日：读 label 一次 + 各因子一次，对齐后逐时点算相关。
    返回 {factor: [(exchtime, pearson, spearman), ...]}（只含与 label 对齐的前 8 个时点）。
    因子的 15:00 收盘行不在 label → 自动丢弃。

    neutralize in {None,"size","industry","both"}：非 None 时，对每个截面把因子
    先对 [1, z(log流通市值), 行业哑元] 回归取残差，再与 label 算相关。市值/行业已在
    expo（暴露缓存）里预算好（含 NAS close），运行期不再读 NAS。
    """
    lex, lsyms, lmat = _load(label_file)
    lrow = {int(e): i for i, e in enumerate(lex)}        # exchtime → label 行号
    lpos = {s: i for i, s in enumerate(lsyms)}           # symbol   → label 列号

    out = {}
    for name, ffile in factor_files.items():
        fex, fsyms, fmat = _load(ffile)
        fpos = {s: i for i, s in enumerate(fsyms)}
        common = [s for s in fsyms if s in lpos]         # 按名取交集（保持因子列序）
        recs = []
        if common:
            fcol = np.fromiter((fpos[s] for s in common), dtype=np.int64)
            lcol = np.fromiter((lpos[s] for s in common), dtype=np.int64)
            exp = (fn.date_exposures(date, common, expo)
                   if neutralize else None)              # 暴露按 common 顺序对齐 fcol
            for r in range(len(fex)):
                e = int(fex[r])
                if e not in lrow:                        # 15:00 收盘行 / label 缺该时点
                    continue
                fvec = fmat[r, fcol]
                if neutralize:                           # 截面残差化（就地对齐 common）
                    fvec = fn.residualize(fvec, exp, neutralize, min_count)
                pear, spear = _corr_pair(fvec, lmat[lrow[e], lcol], min_count)
                recs.append((e, pear, spear))
        out[name] = recs
    return out

## 4. 计算 → `results`（并行按日期）

In [ ]:
from collections import defaultdict
from multiprocessing import Pool

# ---- 找因子 + 构造任务 (date, label_file, {factor: file}) ----
factors = discover_factors(PREFIX, select_factor)
if not factors:
    raise SystemExit(f"在 {FACTOR_ROOT}/{PREFIX} 下没有匹配 select_factor 的因子")

label_dates = set(date_dirs(LABEL_DIR))
lo, hi = (START or "00000000"), (END or "99999999")

fac_files = {name: {d: csv_in(os.path.join(path, d)) for d in date_dirs(path)}
             for name, path in factors.items()}
all_dates = sorted({d for m in fac_files.values() for d in m} & label_dates)

tasks = []
for d in all_dates:
    if not (lo <= d <= hi):
        continue
    lf = csv_in(os.path.join(LABEL_DIR, d))
    if not lf:
        continue
    ff = {name: fac_files[name][d] for name in factors if fac_files[name].get(d)}
    if ff:
        tasks.append((d, lf, ff))

if not tasks:
    raise SystemExit("没有可用的（因子 ∩ label）交易日")

print(f"prefix : {PREFIX}")
print(f"因子   : {', '.join(sorted(factors))}  （{len(factors)} 个）")
print(f"label  : {LABEL_DIR}")
print(f"日期   : {tasks[0][0]} ~ {tasks[-1][0]}  （{len(tasks)} 天）")
print(f"中性化 : {NEUTRALIZE or '无'}"
      + (f"  行业={INDUSTRY_COL}" if NEUTRALIZE in ('industry', 'both') else ""))
print("-" * 60)

# ---- 统一暴露缓存：覆盖则复用、缺则增量补（触库+读NAS 只发生在缺失日）；worker fork 继承 EXPO ----
EXPO = None
if NEUTRALIZE:
    task_dates = [t[0] for t in tasks]
    EXPO = fn.ensure_cache(task_dates, EXPO_CACHE, INDUSTRY_COL)
    print("-" * 60)

# ---- 并行按日期计算（多进程 fork，绕开 GIL），累积 acc[factor][year] = [(exchtime, pear, spear)] ----
acc = {name: defaultdict(list) for name in factors}


def _run(task):
    """单日 worker；多进程 fork 下继承全局 MIN_COUNT / NEUTRALIZE / EXPO / process_date 等。"""
    d, lf, ff = task
    try:
        return d, process_date(d, lf, ff, MIN_COUNT, NEUTRALIZE, EXPO), None
    except Exception as e:                               # noqa: BLE001
        return d, {}, str(e)


errors, done = [], 0
with Pool(N_WORKERS) as pool:
    for d, res, err in pool.imap_unordered(_run, tasks, chunksize=8):
        if err:
            errors.append((d, err))
            continue
        y = d[:4]
        for name, recs in res.items():
            acc[name][y].extend(recs)
        done += 1
        if done % 200 == 0 or done == len(tasks):
            print(f"\r  进度 {done}/{len(tasks)} 天", end="", flush=True)
print("\n" + "-" * 60)
if errors:
    print(f"  {len(errors)} 天读取失败已跳过（例：{errors[0]}）")

In [ ]:
# ---- 逐年聚合成 4 张表：Pearson/Spearman × IC/ICIR（均用前 8 个对齐时点）----
METRICS = {                        # key -> (相关类型, 统计量)
    "pearson_ic":    ("pearson",  "ic"),
    "spearman_ic":   ("spearman", "ic"),
    "pearson_icir":  ("pearson",  "icir"),
    "spearman_icir": ("spearman", "icir"),
}
METRIC_ORDER = ["pearson_ic", "spearman_ic", "pearson_icir", "spearman_icir"]
METRIC_LABEL = {                   # 展示用简洁标题
    "pearson_ic": "Pearson IC", "spearman_ic": "Spearman IC",
    "pearson_icir": "Pearson ICIR", "spearman_icir": "Spearman ICIR",
}


def _agg(recs, which, stat):
    """recs=[(exchtime,pear,spear)]（前 8 个对齐时点全用）。
    which in {pearson, spearman}；stat="ic"=年内截面相关均值，
    "icir"=年内均值 / 年内 std(ddof=1)。"""
    col = 1 if which == "pearson" else 2
    a = np.asarray([t[col] for t in recs], dtype=float)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return np.nan
    m = float(a.mean())
    if stat == "ic":
        return m
    s = float(a.std(ddof=1)) if a.size > 1 else 0.0
    return (m / s) if s > 0 else np.nan


fac_order = sorted(factors)
years = sorted({y for name in factors for y in acc[name]})
year_index = years + (["ALL"] if INCLUDE_ALL else [])

ic_tables = {}
for key, (which, stat) in METRICS.items():
    df = pd.DataFrame(index=year_index, columns=fac_order, dtype=float)
    for name in fac_order:
        for y in years:
            df.at[y, name] = _agg(acc[name][y], which, stat)
        if INCLUDE_ALL:
            allrecs = [t for y in years for t in acc[name][y]]
            df.at["ALL", name] = _agg(allrecs, which, stat)
    ic_tables[key] = df

# 每年参与聚合的时点数（前 8 时点口径；供参考，不作图）
slice_counts = pd.DataFrame(index=years, columns=fac_order, dtype=int)
for name in fac_order:
    for y in years:
        slice_counts.at[y, name] = len(acc[name][y])

results = {
    "prefix": PREFIX,
    "factors": fac_order,
    "label_dir": LABEL_DIR,
    "min_count": MIN_COUNT,
    "neutralize": NEUTRALIZE,      # None / "size" / "industry" / "both"
    "industry_col": INDUSTRY_COL if NEUTRALIZE in ("industry", "both") else None,
    "date_range": [tasks[0][0], tasks[-1][0]],
    "ic": ic_tables,               # 4 张逐年表：pearson_ic / spearman_ic / pearson_icir / spearman_icir
    "slice_counts": slice_counts,
    "raw": acc,                    # acc[factor][year] = [(exchtime, pear, spear)]
}
print("结果已存入 `results`；`ic_tables = results['ic']`（4 张逐年表：IC + ICIR）。"
      + (f"  [中性化={NEUTRALIZE}]" if NEUTRALIZE else ""))
for key in METRIC_ORDER:
    print(f"  ic_tables['{key}']  shape={ic_tables[key].shape}")

## 5. 作图：每个 IC 一张逐年分组柱状图
横坐标 = 年份；每个年份内每个因子一根 bar（紧邻），年份之间留间距。共 4 张图。
颜色按因子固定顺序取自校验过的分类调色板；> 8 个因子时叠加纹理作第二区分通道。

In [ ]:
import matplotlib.pyplot as plt

# 更亮的分类调色板（固定顺序）+ 图表 chrome
_PALETTE = ["#3b8ef0", "#ff7a3d", "#12c9a0", "#ffc21f",
            "#ff7ab6", "#35c72b", "#8b6cff", "#ff5350"]
_HATCHES = [None, "///", "...", "xxx"]      # 因子数 > 8 时用纹理作第二通道
_INK, _MUTED, _GRID, _BASE, _SURF = "#0b0b0b", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
_YLABEL = {"ic": "IC  (yearly mean of cross-sectional corr)",
           "icir": "ICIR  (yearly mean / std)"}


def _factor_style(i):
    return _PALETTE[i % len(_PALETTE)], _HATCHES[(i // len(_PALETTE)) % len(_HATCHES)]


def plot_ic(name, df, ax=None):
    """一张逐年分组柱状图：x=年份，组内一因子一 bar。"""
    facs = list(df.columns)
    xs = [str(x) for x in df.index]                 # 年份 (+ ALL)
    n = max(len(facs), 1)
    if ax is None:
        width = min(26, max(9, 0.42 * n * len(xs) + 3))
        fig, ax = plt.subplots(figsize=(width, 5.0), dpi=130)
        fig.patch.set_facecolor(_SURF)
    ax.set_facecolor(_SURF)

    group_w = 0.82                                  # 一年内 n 根 bar 占的总宽
    bw = group_w / n
    base = np.arange(len(xs))
    for i, fac in enumerate(facs):
        color, hatch = _factor_style(i)
        offs = base - group_w / 2 + bw * (i + 0.5)
        vals = df[fac].to_numpy(dtype=float)
        ax.bar(offs, np.nan_to_num(vals, nan=0.0), width=bw * 0.9,
               color=color, hatch=hatch, edgecolor=_SURF, linewidth=0.6,
               label=fac, zorder=3)

    ax.axhline(0, color=_BASE, lw=1.0, zorder=2)    # 绕 0，画零线
    ax.set_xticks(base)
    ax.set_xticklabels(xs)
    stat = "icir" if name.endswith("icir") else "ic"
    ax.set_ylabel(_YLABEL[stat], color=_INK, fontsize=9)
    ax.set_title(METRIC_LABEL.get(name, name), color=_INK, fontsize=12, pad=8)
    ax.yaxis.grid(True, color=_GRID, lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(_BASE)
    ax.tick_params(colors=_MUTED, labelsize=9)
    ncol = 1 if n <= 14 else 2
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False,
              fontsize=8, ncol=ncol, title="factor", title_fontsize=8,
              labelcolor=_INK)
    return ax


_save_dir = os.environ.get("IC_SAVE_PLOTS")         # 设了就把图存盘（无头验证用）
for name in METRIC_ORDER:
    ax = plot_ic(name, ic_tables[name])
    plt.tight_layout()
    if _save_dir:
        os.makedirs(_save_dir, exist_ok=True)
        ax.figure.savefig(os.path.join(_save_dir, f"ic_{PREFIX}_{name}.png"),
                           bbox_inches="tight", facecolor=_SURF)
    plt.show()

## 6. 打印 markdown 表格
传入一个因子名 list，输出 4 张 markdown 表（每个 IC 一张）：列 = list 里的因子，行 = 年份。

In [ ]:
def _fmt(v):
    return "" if v is None or (isinstance(v, float) and not np.isfinite(v)) else f"{v:+.4f}"


def ic_markdown(name, df, factor_list) -> str:
    """某张表 → markdown 文本：列 = factor_list（保序），行 = 年份。"""
    cols = [c for c in factor_list if c in df.columns]
    missing = [c for c in factor_list if c not in df.columns]
    title = METRIC_LABEL.get(name, name)
    if not cols:                                    # list 里的因子全都不在结果里
        return (f"#### {title}\n\n"
                f"> 注：{missing} 均不在结果里（未被 select_factor 选中或无数据）。")
    head = "| Year | " + " | ".join(cols) + " |"
    sep = "|:-----|" + "|".join(["-------:"] * len(cols)) + "|"
    lines = [f"#### {title}", "", head, sep]
    for y in df.index:
        row = " | ".join(_fmt(df.at[y, c]) for c in cols)
        lines.append(f"| {y} | {row} |")
    md = "\n".join(lines)
    if missing:
        md += f"\n\n> 注：{missing} 不在结果里（未被 select_factor 选中或无数据）。"
    return md


def print_ic_tables(factor_list, ic_order=METRIC_ORDER):
    """打印 4 张 markdown 表的【纯文本】，可直接复制粘贴进 .md 文件。"""
    for name in ic_order:
        print(ic_markdown(name, ic_tables[name], factor_list))
        print()                                     # 表间空行


# 示例：改成你要看的因子 list，例如 ["mean", "twa"]
print_ic_tables(results["factors"])